<a href="https://colab.research.google.com/github/Firefox-097/Steganography/blob/main/Steganography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install kaggle


!mkdir ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json  ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


!kaggle datasets download -d marcozuppelli/stegoimagesdataset
!unzip -q /content/stegoimagesdataset.zip -d /content/stegoimages

mkdir: cannot create directory ‘/root/.kaggle’: File exists
Dataset URL: https://www.kaggle.com/datasets/marcozuppelli/stegoimagesdataset
License(s): DbCL-1.0
stegoimagesdataset.zip: Skipping, found more recently modified local copy (use --force to force download)
replace /content/stegoimages/dataset_information.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "/content/stegoimages/train/train"
val_dir = "/content/stegoimages/val/val"

datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary'
)

val_gen = datagen.flow_from_directory(
    val_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary'
)


Found 16000 images belonging to 2 classes.
Found 8000 images belonging to 2 classes.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(256, 256, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,839,105 (56.61 MB)

 Trainable params: 14,839,105 (56.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen
)


Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 131s 243ms/step - accuracy: 0.7481 - loss: 0.5961 - val_accuracy: 0.7500 - val_loss: 0.5640
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 117s 234ms/step - accuracy: 0.7500 - loss: 0.5697 - val_accuracy: 0.7500 - val_loss: 0.5624
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 119s 237ms/step - accuracy: 0.7500 - loss: 0.5696 - val_accuracy: 0.7500 - val_loss: 0.5624
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 119s 238ms/step - accuracy: 0.7500 - loss: 0.5680 - val_accuracy: 0.7500 - val_loss: 0.5623
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 121s 242ms/step - accuracy: 0.7500 - loss: 0.5673 - val_accuracy: 0.7500 - val_loss: 0.5625
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 122s 243ms/step - accuracy: 0.7500 - loss: 0.5659 - val_accuracy: 0.7500 - val_loss: 0.5643
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 121s 243ms/step - accuracy: 0.7500 - loss: 0.5661 - val_accuracy: 0.7500 - val_loss: 0.5624
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 119s 237ms/step - accuracy: 0.7500 -

In [13]:
model.save("/content/drive/MyDrive/steganography_model.keras")

print("Model saved successfully!")

Model saved successfully!


In [14]:
print("Final Training Accuracy:", history.history['accuracy'][-1])
print("Final Validation Accuracy:", history.history['val_accuracy'][-1])

Final Training Accuracy: 0.75
Final Validation Accuracy: 0.75


In [10]:

from google.colab import files
import shutil

uploaded = files.upload()
original_filename = next(iter(uploaded))
shutil.move(original_filename, "/content/test.jpeg")

from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/test.jpeg"

img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)



Saving Screenshot 2026-03-29 203135.png to Screenshot 2026-03-29 203135.png


In [11]:
prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print("Stego Image")
    secret=True
else:
    print("Cover Image")
    secret=False


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 868ms/step
Stego Image


In [15]:
!pip install stegano

from stegano import lsb

if secret:
    message = lsb.reveal(img_path)

    if message:
        print("Hidden Message:", message)
    else:
        print("No hidden message found")

IndexError: Impossible to detect message.

In [ ]:
from stegano import lsb

secret_message = input("Enter secret message: ")

secret_image = lsb.hide(img_path, secret_message)

secret_image.save("encoded.png")

print("Encoded image saved as encoded.png")